In [ ]:
!pip install rasterio geopandas pandas requests tqdm pyarrow


In [ ]:
import pandas as pd

presence = pd.read_csv("bahgoo_final_occurrences.csv")
background = pd.read_csv("bahgoo_background_points.csv")

print("PRESENCE:")
print(presence.shape)
print(presence.columns.tolist())

print("\nBACKGROUND:")
print(background.shape)
print(background.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'bahgoo_final_occurrences.csv'

In [ ]:
!python 04b_extract_environmental_variables.py \
    --presence bahgoo_final_occurrences.csv \
    --background bahgoo_background_points.csv \
    --code bahgoo \
    --outdir ./aviantrack_env \
    --worldclim-res 2.5m

In [ ]:
import pandas as pd

# Convert presence data
presence = pd.read_csv("bahgoo_final_occurrences.csv")
presence.to_parquet("bahgoo_final_occurrences.parquet", index=False)

# Convert background data
background = pd.read_csv("bahgoo_background_points.csv")
background.to_parquet("bahgoo_background_points.parquet", index=False)

print("Conversion complete!")
print("Presence:", presence.shape)
print("Background:", background.shape)

In [ ]:
import os

for f in os.listdir("."):
    if f.endswith(".parquet"):
        print(f)

In [ ]:
!python 04b_extract_environmental_variables.py \
    --presence bahgoo_final_occurrences.parquet \
    --background bahgoo_background_points.parquet \
    --code bahgoo \
    --outdir ./aviantrack_env \
    --worldclim-res 2.5m

In [ ]:
import pandas as pd

df = pd.read_parquet(
    "aviantrack_env/bahgoo_modeling_dataset.parquet"
)

# Find rows where environmental data is missing
missing = df[df["elevation_m"].isna()].copy()

print("Missing environmental points:", len(missing))
print("\nBy presence/background:")
print(missing["PRESENCE"].value_counts())

print("\nCoordinate range of missing points:")
print("Latitude:", missing["LAT"].min(), "to", missing["LAT"].max())
print("Longitude:", missing["LON"].min(), "to", missing["LON"].max())

print("\nFirst 20 missing coordinates:")
display(missing[["LON", "LAT", "PRESENCE", "SOURCE", "SPECIES_CODE"]].head(20))

In [ ]:
print("Overall coordinate range:")
print("Latitude:", df["LAT"].min(), "to", df["LAT"].max())
print("Longitude:", df["LON"].min(), "to", df["LON"].max())

print("\nInvalid latitude:")
print(((df["LAT"] < -90) | (df["LAT"] > 90)).sum())

print("\nInvalid longitude:")
print(((df["LON"] < -180) | (df["LON"] > 180)).sum())

In [ ]:
import pandas as pd

df = pd.read_parquet(
    "aviantrack_env/bahgoo_modeling_dataset.parquet"
)

# Missing environmental values
missing_presence = df[
    (df["PRESENCE"] == 1) &
    (df["elevation_m"].isna())
].copy()

print("Missing presence records:", len(missing_presence))

display(
    missing_presence[
        ["LON", "LAT", "SOURCE", "SPECIES_CODE"]
    ].sort_values(["LAT", "LON"])
)

In [ ]:
print("Missing presence coordinate ranges")

print(
    "Latitude:",
    missing_presence["LAT"].min(),
    "to",
    missing_presence["LAT"].max()
)

print(
    "Longitude:",
    missing_presence["LON"].min(),
    "to",
    missing_presence["LON"].max()
)

In [ ]:
import glob
import rasterio

tifs = glob.glob("aviantrack_env/**/*.tif", recursive=True)

print("TIFF files found:", len(tifs))

for tif in tifs[:10]:
    with rasterio.open(tif) as src:
        print("\nFile:", tif)
        print("CRS:", src.crs)
        print("Bounds:", src.bounds)
        print("NoData:", src.nodata)
        break

In [ ]:
import pandas as pd

df = pd.read_parquet(
    "aviantrack_env/bahgoo_modeling_dataset.parquet"
)

# Identify missing environmental values
env_cols = [
    "elevation_m",
    "annual_mean_temp",
    "mean_diurnal_range",
    "isothermality",
    "temp_seasonality",
    "max_temp_warmest_month",
    "min_temp_coldest_month",
    "temp_annual_range",
    "mean_temp_wettest_quarter",
    "mean_temp_driest_quarter",
    "mean_temp_warmest_quarter",
    "mean_temp_coldest_quarter",
    "annual_precipitation",
    "precip_wettest_month",
    "precip_driest_month",
    "precip_seasonality",
    "precip_wettest_quarter",
    "precip_driest_quarter",
    "precip_warmest_quarter",
    "precip_coldest_quarter"
]

missing = df[env_cols].isna().any(axis=1)

print("Total missing:", missing.sum())

print("\nBy PRESENCE:")
print(df.loc[missing, "PRESENCE"].value_counts())

print("\nBy SOURCE:")
print(df.loc[missing, "SOURCE"].value_counts())

print("\nMissing percentage:")
print(
    df.loc[missing, "PRESENCE"]
      .value_counts(normalize=True)
      .mul(100)
)

In [ ]:
missing_points = df.loc[
    missing,
    ["LON", "LAT", "PRESENCE", "SOURCE", "SPECIES_CODE"]
].copy()

display(
    missing_points
    .sort_values(["PRESENCE", "LAT", "LON"])
    .head(100)
)

In [ ]:
import pandas as pd
import numpy as np

# Load the modeling dataset
df = pd.read_parquet(
    "aviantrack_env/bahgoo_modeling_dataset.parquet"
)

env_cols = [
    "elevation_m",
    "annual_mean_temp",
    "mean_diurnal_range",
    "isothermality",
    "temp_seasonality",
    "max_temp_warmest_month",
    "min_temp_coldest_month",
    "temp_annual_range",
    "mean_temp_wettest_quarter",
    "mean_temp_driest_quarter",
    "mean_temp_warmest_quarter",
    "mean_temp_coldest_quarter",
    "annual_precipitation",
    "precip_wettest_month",
    "precip_driest_month",
    "precip_seasonality",
    "precip_wettest_quarter",
    "precip_driest_quarter",
    "precip_warmest_quarter",
    "precip_coldest_quarter"
]

# Check how many missing values each environmental variable has
print("Missing values by variable:")
print(df[env_cols].isna().sum())

# Check whether ALL environmental variables are missing together
all_missing = df[env_cols].isna().all(axis=1)

# Check whether ANY environmental variable is missing
any_missing = df[env_cols].isna().any(axis=1)

print("\nAll environmental values missing:", all_missing.sum())
print("At least one environmental value missing:", any_missing.sum())

print("\nBy source:")
print(
    df.loc[any_missing, "SOURCE"]
    .value_counts()
)

print("\nBy presence/background:")
print(
    df.loc[any_missing, "PRESENCE"]
    .value_counts()
)

In [ ]:
missing_presence = df[
    (df["PRESENCE"] == 1) &
    (df[env_cols].isna().any(axis=1))
].copy()

print("Missing presence records:", len(missing_presence))

display(
    missing_presence[
        ["LON", "LAT", "SOURCE", "SPECIES_CODE"] + env_cols
    ]
)

In [ ]:
import glob
import rasterio

tifs = glob.glob(
    "aviantrack_env/**/*.tif",
    recursive=True
)

print("Number of TIFF files:", len(tifs))

for tif in tifs[:25]:
    with rasterio.open(tif) as src:
        print(
            "\nFILE:", tif,
            "\nCRS:", src.crs,
            "\nNoData:", src.nodata,
            "\nBounds:", src.bounds
        )

In [ ]:
import rasterio

# Find elevation raster
elev_file = "aviantrack_env/raw/worldclim/elev/wc2.1_2.5m_elev.tif"

with rasterio.open(elev_file) as src:

    coords = list(
        zip(
            missing_presence["LON"],
            missing_presence["LAT"]
        )
    )

    values = list(src.sample(coords))

    missing_presence["elevation_raw"] = [
        v[0] for v in values
    ]

display(
    missing_presence[
        ["LON", "LAT", "SOURCE", "elevation_raw"]
    ]
)

In [ ]:
import rasterio

tif = "aviantrack_env/raw/worldclim/elev/wc2.1_2.5m_elev.tif"

test_points = [
    (91.377754, 22.392698),
    (80.257594, 12.749178),
    (90.601782, 23.120064),
    (120.967874, 32.852131),
    (73.700000, 15.500000),
]

with rasterio.open(tif) as src:

    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("NoData:", src.nodata)

    print("\nTesting coordinates:\n")

    for lon, lat in test_points:

        value = list(src.sample([(lon, lat)]))[0][0]

        print(
            f"LON={lon:.6f}, LAT={lat:.6f} -> "
            f"ELEVATION={value}"
        )

In [ ]:
with rasterio.open(tif) as src:

    print("Testing REVERSED order:\n")

    for lon, lat in test_points:

        value = list(src.sample([(lat, lon)]))[0][0]

        print(
            f"LAT={lat:.6f}, LON={lon:.6f} -> "
            f"ELEVATION={value}"
        )

In [ ]:
bio_tif = "aviantrack_env/raw/worldclim/bio/wc2.1_2.5m_bio_1.tif"

with rasterio.open(bio_tif) as src:

    print("BIO1 CRS:", src.crs)
    print("BIO1 NoData:", src.nodata)

    for lon, lat in test_points:

        value = list(src.sample([(lon, lat)]))[0][0]

        print(
            f"LON={lon:.6f}, LAT={lat:.6f} -> "
            f"BIO1={value}"
        )

In [ ]:
src.sample([(lon, lat)])

In [ ]:
import rasterio
import numpy as np

tif = "aviantrack_env/raw/worldclim/elev/wc2.1_2.5m_elev.tif"

with rasterio.open(tif) as src:

    print("===== RASTER DIAGNOSTIC =====")
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Shape:", (src.height, src.width))
    print("Dtype:", src.dtypes)
    print("NoData:", src.nodata)
    print("Transform:")
    print(src.transform)

    # Read a small sample of the raster
    data = src.read(1)

    print("\n===== DATA CHECK =====")

    valid = data[data != src.nodata]

    print("Total pixels:", data.size)
    print("NoData pixels:", np.sum(data == src.nodata))
    print("Valid pixels:", len(valid))

    if len(valid) > 0:
        print("Minimum valid elevation:", valid.min())
        print("Maximum valid elevation:", valid.max())
        print("Mean valid elevation:", valid.mean())
    else:
        print("WARNING: ENTIRE RASTER IS NODATA")

In [ ]:
import rasterio
import numpy as np

tif = "aviantrack_env/raw/worldclim/bio/wc2.1_2.5m_bio_1.tif"

with rasterio.open(tif) as src:

    print("===== BIO1 DIAGNOSTIC =====")
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Dtype:", src.dtypes)
    print("NoData:", src.nodata)

    data = src.read(1)

    nodata = src.nodata

    valid = data[data != nodata]

    print("\nTotal pixels:", data.size)
    print("NoData pixels:", np.sum(data == nodata))
    print("Valid pixels:", len(valid))

    if len(valid) > 0:
        print("Minimum valid BIO1:", valid.min())
        print("Maximum valid BIO1:", valid.max())
        print("Mean valid BIO1:", valid.mean())
    else:
        print("WARNING: ENTIRE BIO1 RASTER IS NODATA")

In [ ]:
import rasterio

lon = 91.377754
lat = 22.392698

tif = "aviantrack_env/raw/worldclim/elev/wc2.1_2.5m_elev.tif"

with rasterio.open(tif) as src:

    row, col = src.index(lon, lat)

    print("Point:")
    print("Longitude:", lon)
    print("Latitude:", lat)

    print("\nPixel:")
    print("Row:", row)
    print("Column:", col)

    print("\nPixel value:")
    print(src.read(1)[row, col])

In [ ]:
import rasterio

tif = "aviantrack_env/raw/worldclim/elev/wc2.1_2.5m_elev.tif"

test_points = [
    ("Bangladesh", 90.60, 23.12),
    ("India-Goa", 73.70, 15.50),
    ("Chennai", 80.27, 13.08),
    ("Delhi", 77.21, 28.61),
    ("China", 120.97, 32.85),
    ("London", -0.12, 51.50),
    ("New York", -74.00, 40.71),
]

with rasterio.open(tif) as src:

    print("===== KNOWN LOCATION TEST =====")

    for name, lon, lat in test_points:

        row, col = src.index(lon, lat)
        value = src.read(1)[row, col]

        print(
            f"{name:15s} "
            f"LON={lon:9.4f} "
            f"LAT={lat:8.4f} "
            f"ROW={row:5d} "
            f"COL={col:5d} "
            f"VALUE={value}"
        )

In [ ]:
import rasterio
import numpy as np

tif = "aviantrack_env/raw/worldclim/elev/wc2.1_2.5m_elev.tif"

lon = 90.601782
lat = 23.120064

with rasterio.open(tif) as src:

    row, col = src.index(lon, lat)

    window = src.read(
        1,
        window=rasterio.windows.Window(
            col - 5,
            row - 5,
            11,
            11
        )
    )

    print("Point:", lon, lat)
    print("Pixel:", row, col)

    print("\n11 × 11 neighborhood:")
    print(window)

    valid = window[window != src.nodata]

    print("\nValid pixels nearby:", len(valid))

    if len(valid):
        print("Minimum nearby elevation:", valid.min())
        print("Maximum nearby elevation:", valid.max())
    else:
        print("NO VALID PIXELS WITHIN 5 PIXELS")

In [ ]:
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import xy
from math import radians, sin, cos, sqrt, atan2

# Load modeling dataset
df = pd.read_parquet(
    "aviantrack_env/bahgoo_modeling_dataset.parquet"
)

# Missing environmental values
missing_presence = df[
    (df["PRESENCE"] == 1) &
    (df["elevation_m"].isna())
].copy()

tif = "aviantrack_env/raw/worldclim/elev/wc2.1_2.5m_elev.tif"


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2)**2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2)**2
    )

    return 2 * R * atan2(sqrt(a), sqrt(1 - a))


with rasterio.open(tif) as src:

    data = src.read(1)
    nodata = src.nodata

    results = []

    for idx, row in missing_presence.iterrows():

        lon = row["LON"]
        lat = row["LAT"]

        pixel_row, pixel_col = src.index(lon, lat)

        # Search progressively larger windows
        found = False

        for radius in [1, 2, 3, 5, 10, 20]:

            r0 = max(0, pixel_row - radius)
            r1 = min(src.height, pixel_row + radius + 1)

            c0 = max(0, pixel_col - radius)
            c1 = min(src.width, pixel_col + radius + 1)

            window = data[r0:r1, c0:c1]

            valid_positions = np.argwhere(
                window != nodata
            )

            if len(valid_positions) > 0:

                best_distance = float("inf")
                best_value = None
                best_lat = None
                best_lon = None

                for vr, vc in valid_positions:

                    actual_r = r0 + vr
                    actual_c = c0 + vc

                    value = data[actual_r, actual_c]

                    px_lon, px_lat = xy(
                        src.transform,
                        actual_r,
                        actual_c,
                        offset="center"
                    )

                    distance = haversine_km(
                        lat,
                        lon,
                        px_lat,
                        px_lon
                    )

                    if distance < best_distance:

                        best_distance = distance
                        best_value = value
                        best_lat = px_lat
                        best_lon = px_lon

                results.append({
                    "index": idx,
                    "LON": lon,
                    "LAT": lat,
                    "nearest_elevation": best_value,
                    "nearest_lon": best_lon,
                    "nearest_lat": best_lat,
                    "distance_km": best_distance
                })

                found = True
                break

        if not found:

            results.append({
                "index": idx,
                "LON": lon,
                "LAT": lat,
                "nearest_elevation": np.nan,
                "nearest_lon": np.nan,
                "nearest_lat": np.nan,
                "distance_km": np.nan
            })


result_df = pd.DataFrame(results)

display(
    result_df.sort_values("distance_km")
)

In [ ]:
# Check where the missing background points are

missing_bg = df[
    (df["PRESENCE"] == 0) &
    (df["elevation_m"].isna())
].copy()

print("Missing background points:", len(missing_bg))

print("\nLatitude range:")
print(missing_bg["LAT"].min(), "to", missing_bg["LAT"].max())

print("\nLongitude range:")
print(missing_bg["LON"].min(), "to", missing_bg["LON"].max())

print("\nFirst 30 missing background points:")
display(
    missing_bg[
        ["LON", "LAT", "SOURCE", "SPECIES_CODE"]
    ].head(30)
)

In [ ]:
# Check how many missing background points are near the coast
# using the nearest valid WorldClim elevation cell

print("Number of missing background points:", len(missing_bg))

# Show the first 20
display(
    missing_bg[
        ["LON", "LAT"]
    ].head(20)
)

In [ ]:
# ============================================================
# STEP 1: CHECK VALID BACKGROUND POINTS
# ============================================================

env_cols = [
    "elevation_m",
    "annual_mean_temp",
    "mean_diurnal_range",
    "isothermality",
    "temp_seasonality",
    "max_temp_warmest_month",
    "min_temp_coldest_month",
    "temp_annual_range",
    "mean_temp_wettest_quarter",
    "mean_temp_driest_quarter",
    "mean_temp_warmest_quarter",
    "mean_temp_coldest_quarter",
    "annual_precipitation",
    "precip_wettest_month",
    "precip_driest_month",
    "precip_seasonality",
    "precip_wettest_quarter",
    "precip_driest_quarter",
    "precip_warmest_quarter",
    "precip_coldest_quarter"
]

# Background points
background = df[df["PRESENCE"] == 0].copy()

# A point is environmentally valid only if ALL variables exist
background_valid = background[
    background[env_cols].notna().all(axis=1)
].copy()

background_invalid = background[
    ~background[env_cols].notna().all(axis=1)
].copy()

print("Total background points:", len(background))
print("Valid background points:", len(background_valid))
print("Invalid background points:", len(background_invalid))

print(
    "\nPercentage valid:",
    round(len(background_valid) / len(background) * 100, 2),
    "%"
)

print(
    "Percentage invalid:",
    round(len(background_invalid) / len(background) * 100, 2),
    "%"
)

In [ ]:
# ============================================================
# STEP 2: CHECK PRESENCE RECORDS
# ============================================================

presence = df[df["PRESENCE"] == 1].copy()

presence_valid = presence[
    presence[env_cols].notna().all(axis=1)
].copy()

presence_invalid = presence[
    ~presence[env_cols].notna().all(axis=1)
].copy()

print("Total presence records:", len(presence))
print("Valid presence records:", len(presence_valid))
print("Missing environmental values:", len(presence_invalid))

print("\nPresence records with missing environmental data:")

display(
    presence_invalid[
        ["LON", "LAT", "SOURCE", "SPECIES_CODE"]
    ]
)

In [ ]:
# ============================================================
# STEP 3: CREATE CLEAN MODELING DATASET
# ============================================================

# Environmental variables
env_cols = [
    "elevation_m",
    "annual_mean_temp",
    "mean_diurnal_range",
    "isothermality",
    "temp_seasonality",
    "max_temp_warmest_month",
    "min_temp_coldest_month",
    "temp_annual_range",
    "mean_temp_wettest_quarter",
    "mean_temp_driest_quarter",
    "mean_temp_warmest_quarter",
    "mean_temp_coldest_quarter",
    "annual_precipitation",
    "precip_wettest_month",
    "precip_driest_month",
    "precip_seasonality",
    "precip_wettest_quarter",
    "precip_driest_quarter",
    "precip_warmest_quarter",
    "precip_coldest_quarter"
]

# A row is usable only if ALL environmental variables exist
valid_environment = df[env_cols].notna().all(axis=1)

# Keep valid rows only
model_df = df[valid_environment].copy()

print("================================================")
print("CLEAN MODELING DATASET")
print("================================================")

print("Original rows:", len(df))
print("Rows removed:", len(df) - len(model_df))
print("Final rows:", len(model_df))

print("\nBy presence/background:")
print(model_df["PRESENCE"].value_counts())

print("\nBy source:")
print(model_df["SOURCE"].value_counts())

print("\nMissing environmental values remaining:")
print(model_df[env_cols].isna().sum().sum())

In [ ]:
# ============================================================
# STEP 4: SAVE CLEAN DATASET
# ============================================================

import os

os.makedirs("aviantrack_env/clean", exist_ok=True)

output_file = "aviantrack_env/clean/bahgoo_modeling_dataset_clean.parquet"

model_df.to_parquet(
    output_file,
    index=False
)

print("Saved:")
print(output_file)

print("\nFile rows:", len(model_df))
print("File columns:", len(model_df.columns))

In [ ]:
csv_file = "aviantrack_env/clean/bahgoo_modeling_dataset_clean.csv"

model_df.to_csv(
    csv_file,
    index=False
)

print("CSV saved:")
print(csv_file)

In [ ]:
def prepare_species(species_code):
    print("Preparing species:", species_code)

    # We will add the actual pipeline here step by step.

    return None


# Test the function
prepare_species("bahgoo")

In [ ]:
bahgoo = prepare_species("bahgoo")
species_2 = prepare_species("...")
species_3 = prepare_species("...")

In [ ]:
print("Columns in global dataset:")
print(df.columns.tolist())

print("\nNumber of unique species:")
print(df["SPECIES_CODE"].nunique())

print("\nFirst 20 species:")
print(df["SPECIES_CODE"].value_counts().head(20))


In [ ]:
def prepare_species(species_code):

    # Select only the requested species
    species_df = df[df["SPECIES_CODE"] == species_code].copy()

    print("Species:", species_code)
    print("Records:", len(species_df))

    return species_df

In [ ]:
bahgoo = prepare_species("bahgoo")

In [ ]:
species_2 = prepare_species("REAL_SPECIES_CODE")
species_3 = prepare_species("REAL_SPECIES_CODE")

In [ ]:
# ============================================
# STEP 1: PREPARE DATA FOR MACHINE LEARNING
# ============================================

# Environmental variables we will use as features
features = [
    "elevation_m",
    "annual_mean_temp",
    "mean_diurnal_range",
    "isothermality",
    "temp_seasonality",
    "max_temp_warmest_month",
    "min_temp_coldest_month",
    "temp_annual_range",
    "mean_temp_wettest_quarter",
    "mean_temp_driest_quarter",
    "mean_temp_warmest_quarter",
    "mean_temp_coldest_quarter",
    "annual_precipitation",
    "precip_wettest_month",
    "precip_driest_month",
    "precip_seasonality",
    "precip_wettest_quarter",
    "precip_driest_quarter",
    "precip_warmest_quarter",
    "precip_coldest_quarter"
]

# X = environmental information
X = model_df[features].copy()

# y = what we want the model to predict
# 1 = bahgoo was observed
# 0 = background location
y = model_df["PRESENCE"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass distribution:")
print(y.value_counts())

print("\nMissing values:")
print(X.isna().sum().sum())

In [ ]:
# ============================================
# STEP 2: TRAIN / TEST SPLIT
# ============================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

In [ ]:
# ============================================
# STEP 3: TRAIN RANDOM FOREST MODEL
# ============================================

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

print("Training Random Forest...")

model.fit(X_train, y_train)

print("Training completed successfully!")

In [ ]:
# ============================================
# STEP 4: EVALUATE RANDOM FOREST
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions on the unseen test data
y_pred = model.predict(X_test)

# Probability of presence (class 1)
y_prob = model.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("============================================")
print("RANDOM FOREST PERFORMANCE")
print("============================================")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# ============================================
# STEP 5: FEATURE IMPORTANCE
# ============================================

import pandas as pd

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("Top environmental variables:")
print(importance.head(20))

In [ ]:
import matplotlib.pyplot as plt

top = importance.head(10)

plt.figure(figsize=(10, 6))
plt.barh(top["Feature"][::-1], top["Importance"][::-1])
plt.xlabel("Feature Importance")
plt.ylabel("Environmental Variable")
plt.title("Top Environmental Variables - Bahgoo Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# STEP 1: CREATE SPATIAL GROUPS
# ============================================

import pandas as pd
import numpy as np

# Load the clean Bahgoo dataset
clean_file = "aviantrack_env/clean/bahgoo_modeling_dataset_clean.parquet"

df_spatial = pd.read_parquet(clean_file)

print("Dataset shape:", df_spatial.shape)
print("\nColumns:")
print(df_spatial.columns.tolist())

print("\nCoordinate range:")
print("Latitude :", df_spatial["LAT"].min(), "to", df_spatial["LAT"].max())
print("Longitude:", df_spatial["LON"].min(), "to", df_spatial["LON"].max())

In [ ]:
# ============================================
# STEP 2: CREATE SPATIAL BLOCKS
# ============================================

# Size of each spatial block in degrees
BLOCK_SIZE = 2.0

df_spatial["spatial_block_lat"] = (
    np.floor(df_spatial["LAT"] / BLOCK_SIZE)
).astype(int)

df_spatial["spatial_block_lon"] = (
    np.floor(df_spatial["LON"] / BLOCK_SIZE)
).astype(int)

# Create a unique block ID
df_spatial["spatial_block"] = (
    df_spatial["spatial_block_lat"].astype(str)
    + "_"
    + df_spatial["spatial_block_lon"].astype(str)
)

print("Total records:", len(df_spatial))
print("Number of spatial blocks:", df_spatial["spatial_block"].nunique())

print("\nRecords per spatial block:")
print(
    df_spatial["spatial_block"]
    .value_counts()
    .describe()
)

print("\nFirst 10 spatial blocks:")
print(
    df_spatial[
        ["LON", "LAT", "spatial_block"]
    ].head(10)
)

In [ ]:
# ============================================
# STEP 3: SELECT SPATIAL TRAIN / TEST BLOCKS
# ============================================

from sklearn.model_selection import train_test_split

# Get information about each spatial block
block_info = (
    df_spatial
    .groupby("spatial_block")["PRESENCE"]
    .agg(["count", "sum"])
    .reset_index()
)

# Number of background records in each block
block_info["background"] = (
    block_info["count"] - block_info["sum"]
)

# Identify blocks containing both classes
mixed_blocks = block_info[
    (block_info["sum"] > 0) &
    (block_info["background"] > 0)
]["spatial_block"].tolist()

print("Total spatial blocks:", len(block_info))
print("Blocks containing both classes:", len(mixed_blocks))

# Split the mixed blocks geographically
train_blocks, test_blocks = train_test_split(
    mixed_blocks,
    test_size=0.20,
    random_state=42
)

print("\nTraining blocks:", len(train_blocks))
print("Testing blocks:", len(test_blocks))

# Create train/test datasets
spatial_train = df_spatial[
    df_spatial["spatial_block"].isin(train_blocks)
].copy()

spatial_test = df_spatial[
    df_spatial["spatial_block"].isin(test_blocks)
].copy()

print("\nTraining records:", len(spatial_train))
print("Testing records:", len(spatial_test))

print("\nTraining class distribution:")
print(spatial_train["PRESENCE"].value_counts())

print("\nTesting class distribution:")
print(spatial_test["PRESENCE"].value_counts())

In [ ]:
# ============================================
# STEP 4: SPATIAL RANDOM FOREST
# ============================================

from sklearn.ensemble import RandomForestClassifier

# Remove the spatial helper columns
feature_cols = [
    col for col in spatial_train.columns
    if col not in [
        "LON",
        "LAT",
        "PRESENCE",
        "SOURCE",
        "SPECIES_CODE",
        "spatial_block_lat",
        "spatial_block_lon",
        "spatial_block"
    ]
]

X_spatial_train = spatial_train[feature_cols]
y_spatial_train = spatial_train["PRESENCE"]

X_spatial_test = spatial_test[feature_cols]
y_spatial_test = spatial_test["PRESENCE"]

print("Features:", len(feature_cols))
print("X_train:", X_spatial_train.shape)
print("X_test :", X_spatial_test.shape)

# Train model
spatial_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

print("\nTraining spatial Random Forest...")

spatial_model.fit(
    X_spatial_train,
    y_spatial_train
)

print("Spatial Random Forest training completed!")

In [ ]:
# ============================================
# STEP 5: EVALUATE SPATIAL RANDOM FOREST
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions on unseen geographic blocks
y_spatial_pred = spatial_model.predict(X_spatial_test)

# Probability of presence
y_spatial_prob = spatial_model.predict_proba(
    X_spatial_test
)[:, 1]

# Calculate metrics
spatial_accuracy = accuracy_score(
    y_spatial_test,
    y_spatial_pred
)

spatial_precision = precision_score(
    y_spatial_test,
    y_spatial_pred
)

spatial_recall = recall_score(
    y_spatial_test,
    y_spatial_pred
)

spatial_f1 = f1_score(
    y_spatial_test,
    y_spatial_pred
)

spatial_auc = roc_auc_score(
    y_spatial_test,
    y_spatial_prob
)

print("============================================")
print("SPATIAL RANDOM FOREST PERFORMANCE")
print("============================================")

print(f"Accuracy : {spatial_accuracy:.4f}")
print(f"Precision: {spatial_precision:.4f}")
print(f"Recall   : {spatial_recall:.4f}")
print(f"F1 Score : {spatial_f1:.4f}")
print(f"ROC-AUC  : {spatial_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_spatial_test,
    y_spatial_pred
))

print("\nClassification Report:")
print(classification_report(
    y_spatial_test,
    y_spatial_pred
))

In [ ]:
# ============================================
# STEP 6: ANALYZE SPATIAL TEST PERFORMANCE
# ============================================

spatial_results = spatial_test[
    ["LON", "LAT", "PRESENCE", "spatial_block"]
].copy()

spatial_results["PREDICTED"] = y_spatial_pred
spatial_results["PRESENCE_PROBABILITY"] = y_spatial_prob

# Correct / incorrect prediction
spatial_results["CORRECT"] = (
    spatial_results["PRESENCE"] ==
    spatial_results["PREDICTED"]
)

print("Total spatial test records:",
      len(spatial_results))

print("\nCorrect predictions:",
      spatial_results["CORRECT"].sum())

print("Incorrect predictions:",
      (~spatial_results["CORRECT"]).sum())

print("\nAccuracy by spatial block:")

block_accuracy = (
    spatial_results
    .groupby("spatial_block")
    .agg(
        records=("PRESENCE", "size"),
        actual_presence=("PRESENCE", "sum"),
        accuracy=("CORRECT", "mean"),
        mean_probability=("PRESENCE_PROBABILITY", "mean")
    )
    .sort_values("accuracy")
)

print(block_accuracy.head(20))

In [ ]:
# ============================================
# STEP 7: VISUALIZE SPATIAL PERFORMANCE
# ============================================

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))

# Correct predictions
correct = spatial_results[spatial_results["CORRECT"] == True]

# Incorrect predictions
incorrect = spatial_results[spatial_results["CORRECT"] == False]

plt.scatter(
    correct["LON"],
    correct["LAT"],
    s=5,
    alpha=0.3,
    label="Correct"
)

plt.scatter(
    incorrect["LON"],
    incorrect["LAT"],
    s=5,
    alpha=0.3,
    label="Incorrect"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Bahgoo Spatial Validation: Correct vs Incorrect Predictions")
plt.legend()
plt.grid(alpha=0.2)

plt.show()

In [ ]:
# ============================================
# STEP 1: PREDICTION PROBABILITY ANALYSIS
# ============================================

# Get probability of PRESENCE = 1
spatial_prob = spatial_rf.predict_proba(X_test)[:, 1]

print("Number of test predictions:", len(spatial_prob))
print()
print("Minimum probability:", spatial_prob.min())
print("Maximum probability:", spatial_prob.max())
print("Mean probability:", spatial_prob.mean())
print("Median probability:", np.median(spatial_prob))

print("\nProbability ranges:")

bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5,
        0.6, 0.7, 0.8, 0.9, 1.0]

prob_counts = pd.cut(
    spatial_prob,
    bins=bins,
    include_lowest=True
).value_counts().sort_index()

print(prob_counts)

In [ ]:
# Find variables containing "rf" or "forest"
[name for name in globals()
 if "rf" in name.lower() or "forest" in name.lower()]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Find trained Random Forest model objects
rf_models = []

for name, obj in globals().items():
    if isinstance(obj, RandomForestClassifier):
        rf_models.append(name)

print("Trained Random Forest model variables:")
print(rf_models)

In [ ]:
# ============================================
# STEP 1: PREDICTION PROBABILITY ANALYSIS
# ============================================

# Get probability of PRESENCE = 1
spatial_prob = spatial_model.predict_proba(X_test)[:, 1]

print("Number of test predictions:", len(spatial_prob))
print()

print("Minimum probability:", spatial_prob.min())
print("Maximum probability:", spatial_prob.max())
print("Mean probability:", spatial_prob.mean())
print("Median probability:", np.median(spatial_prob))

print("\nProbability ranges:")

bins = [
    0, 0.1, 0.2, 0.3, 0.4, 0.5,
    0.6, 0.7, 0.8, 0.9, 1.0
]

prob_counts = (
    pd.cut(
        spatial_prob,
        bins=bins,
        include_lowest=True
    )
    .value_counts()
    .sort_index()
)

print(prob_counts)

In [ ]:
# ============================================
# STEP 2: THRESHOLD ANALYSIS
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

thresholds = np.arange(0.20, 0.81, 0.05)

threshold_results = []

for threshold in thresholds:

    y_pred_threshold = (
        spatial_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(
            y_test,
            y_pred_threshold
        ),
        "Precision": precision_score(
            y_test,
            y_pred_threshold,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            y_pred_threshold,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            y_pred_threshold,
            zero_division=0
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_test,
            y_pred_threshold
        )
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.round(4)
)

In [ ]:
# Find variables related to train/test datasets
[
    name for name in globals()
    if any(word in name.lower()
           for word in ["x_train", "x_test", "y_train", "y_test", "spatial"])
]

In [ ]:
# ============================================
# STEP 2: CORRECT SPATIAL THRESHOLD ANALYSIS
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)
import numpy as np
import pandas as pd

# Correct spatial test probabilities
spatial_prob = spatial_model.predict_proba(
    X_spatial_test
)[:, 1]

print("Number of spatial test predictions:",
      len(spatial_prob))

print("Number of spatial test labels:",
      len(y_spatial_test))

# Check that they match
assert len(spatial_prob) == len(y_spatial_test)

thresholds = np.arange(0.20, 0.81, 0.05)

threshold_results = []

for threshold in thresholds:

    y_pred_threshold = (
        spatial_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),

        "Accuracy": accuracy_score(
            y_spatial_test,
            y_pred_threshold
        ),

        "Precision": precision_score(
            y_spatial_test,
            y_pred_threshold,
            zero_division=0
        ),

        "Recall": recall_score(
            y_spatial_test,
            y_pred_threshold,
            zero_division=0
        ),

        "F1": f1_score(
            y_spatial_test,
            y_pred_threshold,
            zero_division=0
        ),

        "Balanced_Accuracy": balanced_accuracy_score(
            y_spatial_test,
            y_pred_threshold
        )
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.round(4)
)

In [ ]:
# ============================================
# STEP 3: SPATIAL MODEL FEATURE IMPORTANCE
# ============================================

import pandas as pd
import matplotlib.pyplot as plt

feature_importance_spatial = pd.DataFrame({
    "Feature": X_spatial_train.columns,
    "Importance": spatial_model.feature_importances_
})

feature_importance_spatial = (
    feature_importance_spatial
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

print("Environmental variables used by Spatial Random Forest:")
display(feature_importance_spatial.round(6))

In [ ]:
# ============================================
# TOP 10 ENVIRONMENTAL VARIABLES
# ============================================

top10_spatial = feature_importance_spatial.head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    top10_spatial["Feature"][::-1],
    top10_spatial["Importance"][::-1]
)

plt.xlabel("Feature Importance")
plt.ylabel("Environmental Variable")
plt.title(
    "Top Environmental Variables - Spatial Random Forest"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# STEP 4: 5-FOLD SPATIAL CROSS-VALIDATION
# ============================================

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score
)
import pandas as pd
import numpy as np

# Features and target
X = df_spatial[X_spatial_train.columns]
y = df_spatial["PRESENCE"]

# Spatial groups
groups = df_spatial["spatial_block"]

print("Dataset shape:", X.shape)
print("Number of spatial blocks:", groups.nunique())

# 5-fold spatial cross-validation
gkf = GroupKFold(n_splits=5)

cv_results = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups=groups), start=1
):

    print("\n" + "=" * 45)
    print(f"SPATIAL FOLD {fold}")
    print("=" * 45)

    X_train_fold = X.iloc[train_idx]
    X_test_fold = X.iloc[test_idx]

    y_train_fold = y.iloc[train_idx]
    y_test_fold = y.iloc[test_idx]

    print("Training records:", len(train_idx))
    print("Testing records :", len(test_idx))

    print("Training blocks:", groups.iloc[train_idx].nunique())
    print("Testing blocks :", groups.iloc[test_idx].nunique())

    # Train model
    fold_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    fold_model.fit(X_train_fold, y_train_fold)

    # Predictions
    y_pred_fold = fold_model.predict(X_test_fold)
    y_prob_fold = fold_model.predict_proba(X_test_fold)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test_fold, y_pred_fold)
    precision = precision_score(y_test_fold, y_pred_fold, zero_division=0)
    recall = recall_score(y_test_fold, y_pred_fold, zero_division=0)
    f1 = f1_score(y_test_fold, y_pred_fold, zero_division=0)
    balanced_acc = balanced_accuracy_score(y_test_fold, y_pred_fold)
    auc = roc_auc_score(y_test_fold, y_prob_fold)

    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"ROC-AUC            : {auc:.4f}")

    cv_results.append({
        "Fold": fold,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Balanced_Accuracy": balanced_acc,
        "ROC_AUC": auc
    })

# Convert results to DataFrame
cv_results_df = pd.DataFrame(cv_results)

print("\n")
print("=" * 60)
print("5-FOLD SPATIAL CROSS-VALIDATION RESULTS")
print("=" * 60)

display(cv_results_df.round(4))

In [ ]:
# ============================================
# CROSS-VALIDATION SUMMARY
# ============================================

summary = pd.DataFrame({
    "Mean": cv_results_df.drop(columns="Fold").mean(),
    "Std": cv_results_df.drop(columns="Fold").std()
})

print("Average performance across 5 spatial folds:")
display(summary.round(4))

In [ ]:
# ============================================
# STEP 5: ANALYZE SPATIAL CV FOLDS
# ============================================

print("Spatial Cross-Validation Summary")
print("=" * 60)

display(cv_results_df.round(4))

print("\nBest fold by ROC-AUC:")
best_fold = cv_results_df.loc[
    cv_results_df["ROC_AUC"].idxmax()
]
display(best_fold)

print("\nWorst fold by ROC-AUC:")
worst_fold = cv_results_df.loc[
    cv_results_df["ROC_AUC"].idxmin()
]
display(worst_fold)

print("\nBest fold ROC-AUC:",
      round(best_fold["ROC_AUC"], 4))

print("Worst fold ROC-AUC:",
      round(worst_fold["ROC_AUC"], 4))

print("ROC-AUC difference:",
      round(
          best_fold["ROC_AUC"] - worst_fold["ROC_AUC"],
          4
      ))

In [ ]:
# ============================================
# STEP 6: CHECK ENVIRONMENTAL RASTER LAYERS
# ============================================

import rasterio
import os

raster_files = [
    "aviantack_env/raw/worldclim/elev/wc2.1_2.5m_elev.tif",
    "aviantack_env/raw/worldclim/bio/wc2.1_2.5m_bio_1.tif",
    "aviantack_env/raw/worldclim/bio/wc2.1_2.5m_bio_2.tif",
    "aviantack_env/raw/worldclim/bio/wc2.1_2.5m_bio_4.tif",
    "aviantack_env/raw/worldclim/bio/wc2.1_2.5m_bio_12.tif"
]

for file in raster_files:

    print("\n" + "=" * 60)
    print("FILE:", file)
    print("=" * 60)

    if os.path.exists(file):

        with rasterio.open(file) as src:

            print("CRS       :", src.crs)
            print("Width     :", src.width)
            print("Height    :", src.height)
            print("Bands     :", src.count)
            print("Resolution:", src.res)
            print("Bounds    :", src.bounds)
            print("NoData    :", src.nodata)

    else:
        print("FILE NOT FOUND")

In [ ]:
# ============================================
# STEP 6A: FIND THE WORLDCLIM RASTER FILES
# ============================================

import os

search_root = "aviantack_env"

print("Searching for .tif files...\n")

tif_files = []

for root, dirs, files in os.walk(search_root):
    for file in files:
        if file.lower().endswith(".tif"):
            tif_files.append(os.path.join(root, file))

print("Total TIFF files found:", len(tif_files))

for f in tif_files[:100]:
    print(f)

In [ ]:
# ============================================
# STEP 6B: FIND ELEVATION + BIOCLIM FILES
# ============================================

print("\n===== ELEVATION FILES =====")

for f in tif_files:
    if "elev" in os.path.basename(f).lower():
        print(f)

print("\n===== BIOCLIM FILES =====")

for f in tif_files:
    if "bio" in os.path.basename(f).lower():
        print(f)

In [ ]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles/folders in current directory:")
for item in os.listdir("."):
    print(item)

In [ ]:
import os

print("Searching entire /content for TIFF files...\n")

tif_files = []

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.lower().endswith((".tif", ".tiff")):
            tif_files.append(os.path.join(root, file))

print("Total TIFF files found:", len(tif_files))

for f in tif_files[:100]:
    print(f)

In [ ]:
# ============================================
# STEP 6C: VERIFY ALL 20 ENVIRONMENTAL RASTERS
# ============================================

import os
import rasterio

raster_dir = "/content/aviantrack_env/raw/worldclim"

# Expected files
elevation_file = os.path.join(
    raster_dir, "elev", "wc2.1_2.5m_elev.tif"
)

bio_dir = os.path.join(raster_dir, "bio")

bio_files = [
    os.path.join(bio_dir, f"wc2.1_2.5m_bio_{i}.tif")
    for i in range(1, 20)
]

all_files = [elevation_file] + bio_files

print("Expected environmental rasters:", len(all_files))
print()

missing = []

for f in all_files:
    if os.path.exists(f):
        print("✅", os.path.basename(f))
    else:
        print("❌ MISSING:", f)
        missing.append(f)

print("\n--------------------------------------------")

if len(missing) == 0:
    print("SUCCESS: All 20 environmental rasters found!")
else:
    print("Missing files:", len(missing))

In [ ]:
# ============================================
# STEP 6D: CHECK RASTER CONSISTENCY
# ============================================

reference = None
consistent = True

for f in all_files:

    with rasterio.open(f) as src:

        info = {
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "transform": src.transform,
            "bounds": src.bounds,
            "resolution": src.res
        }

        if reference is None:
            reference = info
            print("Reference raster:")
            print(os.path.basename(f))
            print("CRS:", info["crs"])
            print("Size:", info["width"], "x", info["height"])
            print("Resolution:", info["resolution"])
            print("Bounds:", info["bounds"])

        else:

            if (
                info["crs"] != reference["crs"]
                or info["width"] != reference["width"]
                or info["height"] != reference["height"]
                or info["transform"] != reference["transform"]
            ):
                consistent = False
                print(
                    "❌ Inconsistent:",
                    os.path.basename(f)
                )

print("\n--------------------------------------------")

if consistent:
    print("✅ ALL 20 RASTERS ARE CONSISTENT")
else:
    print("⚠️ Some rasters are inconsistent")

In [ ]:
# ============================================
# STEP 7: PREPARE ENVIRONMENTAL RASTERS
# ============================================

import os
import rasterio
import numpy as np

raster_dir = "/content/aviantrack_env/raw/worldclim"

# Model feature order
feature_names = [
    "elevation_m",
    "annual_mean_temp",
    "mean_diurnal_range",
    "isothermality",
    "temp_seasonality",
    "max_temp_warmest_month",
    "min_temp_coldest_month",
    "temp_annual_range",
    "mean_temp_wettest_quarter",
    "mean_temp_driest_quarter",
    "mean_temp_warmest_quarter",
    "mean_temp_coldest_quarter",
    "annual_precipitation",
    "precip_wettest_month",
    "precip_driest_month",
    "precip_seasonality",
    "precip_wettest_quarter",
    "precip_driest_quarter",
    "precip_warmest_quarter",
    "precip_coldest_quarter"
]

# Corresponding raster files
raster_paths = {
    "elevation_m":
        os.path.join(
            raster_dir,
            "elev",
            "wc2.1_2.5m_elev.tif"
        )
}

# BIO1 -> annual_mean_temp
# BIO2 -> mean_diurnal_range
# ...
# BIO19 -> precip_coldest_quarter

for bio_number, feature in enumerate(feature_names[1:], start=1):

    raster_paths[feature] = os.path.join(
        raster_dir,
        "bio",
        f"wc2.1_2.5m_bio_{bio_number}.tif"
    )

print("Environmental variable → raster mapping")
print("=" * 70)

for i, feature in enumerate(feature_names, start=1):

    path = raster_paths[feature]

    print(
        f"{i:2d}. {feature:30s} → "
        f"{os.path.basename(path)}"
    )

    if not os.path.exists(path):
        print("    ❌ FILE NOT FOUND")

print("\n--------------------------------------------")
print("Total variables:", len(feature_names))
print("Total raster files:", len(raster_paths))

if all(os.path.exists(p) for p in raster_paths.values()):
    print("✅ All 20 model variables mapped successfully!")
else:
    print("❌ One or more raster files are missing.")

In [ ]:
!python 04c_add_landcover.py \
  --modeling-dataset ./aviantrack_env/bahgoo_modeling_dataset.parquet \
  --code bahgoo \
  --outdir ./aviantrack_env

In [ ]:
import os

input_file = "/content/aviantrack_env/bahgoo_modeling_dataset.parquet"

print("Checking:", input_file)

if os.path.exists(input_file):
    print("✅ Modeling dataset found")
    print("File size:", round(os.path.getsize(input_file) / (1024**2), 2), "MB")
else:
    print("❌ Modeling dataset NOT found")


In [ ]:
!python 04c_add_landcover.py \
  --modeling-dataset ./aviantrack_env/bahgoo_modeling_dataset.parquet \
  --code bahgoo \
  --outdir ./aviantrack_env

In [ ]:
import pandas as pd

df_landcover = pd.read_parquet(
    "./aviantrack_env/bahgoo_modeling_dataset.parquet"
)

print("Dataset shape:", df_landcover.shape)

print("\nColumns:")
print(df_landcover.columns.tolist())

print("\nLand-cover class distribution:")
print(df_landcover["land_cover_class"].value_counts(dropna=False))

In [ ]:
print("Missing land cover by PRESENCE:")
print(
    df_landcover[df_landcover["land_cover_class"].isna()]
    ["PRESENCE"]
    .value_counts()
)

print("\nMissing land cover by SOURCE:")
print(
    df_landcover[df_landcover["land_cover_class"].isna()]
    ["SOURCE"]
    .value_counts()
)

In [ ]:
print("\nLand-cover codes:")
print(
    df_landcover["land_cover_code"]
    .value_counts(dropna=False)
)

In [ ]:
print("\nMissing land-cover examples:")
print(
    df_landcover[
        df_landcover["land_cover_class"].isna()
    ][["LON", "LAT", "PRESENCE", "SOURCE", "land_cover_code", "land_cover_class"]]
    .head(20)
)

In [ ]:
environmental_cols = [
    "elevation_m",
    "annual_mean_temp",
    "mean_diurnal_range",
    "isothermality",
    "temp_seasonality",
    "max_temp_warmest_month",
    "min_temp_coldest_month",
    "temp_annual_range",
    "mean_temp_wettest_quarter",
    "mean_temp_driest_quarter",
    "mean_temp_warmest_quarter",
    "mean_temp_coldest_quarter",
    "annual_precipitation",
    "precip_wettest_month",
    "precip_driest_month",
    "precip_seasonality",
    "precip_wettest_quarter",
    "precip_driest_quarter",
    "precip_warmest_quarter",
    "precip_coldest_quarter"
]

valid_environment = df_landcover[environmental_cols].notna().all(axis=1)

missing_landcover = df_landcover["land_cover_code"].isna() | (
    df_landcover["land_cover_code"] == 0
)

print("Total records:", len(df_landcover))

print("Records with valid environmental data:",
      valid_environment.sum())

print("Records with valid environment + missing land cover:",
      (valid_environment & missing_landcover).sum())

print("\nBreakdown by PRESENCE:")
print(
    df_landcover[
        valid_environment & missing_landcover
    ]["PRESENCE"].value_counts()
)

In [ ]:
import pandas as pd

# Environmental variables
environmental_cols = [
    "elevation_m",
    "annual_mean_temp",
    "mean_diurnal_range",
    "isothermality",
    "temp_seasonality",
    "max_temp_warmest_month",
    "min_temp_coldest_month",
    "temp_annual_range",
    "mean_temp_wettest_quarter",
    "mean_temp_driest_quarter",
    "mean_temp_warmest_quarter",
    "mean_temp_coldest_quarter",
    "annual_precipitation",
    "precip_wettest_month",
    "precip_driest_month",
    "precip_seasonality",
    "precip_wettest_quarter",
    "precip_driest_quarter",
    "precip_warmest_quarter",
    "precip_coldest_quarter"
]

# Keep records with complete environmental data
df_final = df_landcover[
    df_landcover[environmental_cols].notna().all(axis=1)
].copy()

# Keep only records with valid land-cover code
df_final = df_final[
    df_final["land_cover_code"].notna() &
    (df_final["land_cover_code"] != 0)
].copy()

print("Final dataset shape:", df_final.shape)

print("\nMissing values:")
print(df_final.isna().sum())

print("\nClass distribution:")
print(df_final["PRESENCE"].value_counts())

print("\nLand-cover distribution:")
print(df_final["land_cover_class"].value_counts())

In [ ]:
# ============================================
# STEP 1: PREPARE FEATURES FOR MODELING
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# 20 environmental variables
environmental_cols = [
    "elevation_m",
    "annual_mean_temp",
    "mean_diurnal_range",
    "isothermality",
    "temp_seasonality",
    "max_temp_warmest_month",
    "min_temp_coldest_month",
    "temp_annual_range",
    "mean_temp_wettest_quarter",
    "mean_temp_driest_quarter",
    "mean_temp_warmest_quarter",
    "mean_temp_coldest_quarter",
    "annual_precipitation",
    "precip_wettest_month",
    "precip_driest_month",
    "precip_seasonality",
    "precip_wettest_quarter",
    "precip_driest_quarter",
    "precip_warmest_quarter",
    "precip_coldest_quarter"
]

# Land-cover feature
landcover_col = ["land_cover_class"]

# Target
target_col = "PRESENCE"

# Features and target
X = df_final[environmental_cols + landcover_col]
y = df_final[target_col]

print("============================================")
print("FEATURE PREPARATION")
print("============================================")

print("Dataset shape:", X.shape)
print("Number of features before encoding:", X.shape[1])

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget shape:", y.shape)

print("\nClass distribution:")
print(y.value_counts())

print("\nLand-cover classes:")
print(X["land_cover_class"].value_counts())

In [ ]:
# ============================================
# STEP 2: ENCODE LAND COVER
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        (
            "land_cover",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["land_cover_class"]
        )
    ],
    remainder="passthrough"
)

# Transform features
X_encoded = preprocessor.fit_transform(X)

# Get feature names
encoded_feature_names = preprocessor.get_feature_names_out()

print("============================================")
print("LAND COVER ENCODING")
print("============================================")

print("Original features:", X.shape[1])
print("Encoded features:", X_encoded.shape[1])

print("\nEncoded feature names:")
print(encoded_feature_names.tolist())

print("\nEncoded X shape:", X_encoded.shape)

In [ ]:
# ============================================
# STEP 3: TRAIN / TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("============================================")
print("TRAIN / TEST SPLIT")
print("============================================")

print("Training data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

In [ ]:
# ============================================
# STEP 4: TRAIN RANDOM FOREST
# ============================================

from sklearn.ensemble import RandomForestClassifier

print("============================================")
print("TRAINING RANDOM FOREST")
print("============================================")

rf_landcover = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_landcover.fit(X_train, y_train)

print("\nRandom Forest training completed successfully!")

In [ ]:
# ============================================
# STEP 5: RANDOM FOREST EVALUATION
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_pred_landcover = rf_landcover.predict(X_test)

# Probability of presence
y_prob_landcover = rf_landcover.predict_proba(X_test)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred_landcover)
precision = precision_score(y_test, y_pred_landcover)
recall = recall_score(y_test, y_pred_landcover)
f1 = f1_score(y_test, y_pred_landcover)
balanced_acc = balanced_accuracy_score(y_test, y_pred_landcover)
roc_auc = roc_auc_score(y_test, y_prob_landcover)

print("============================================")
print("RANDOM FOREST + LAND COVER PERFORMANCE")
print("============================================")

print(f"Accuracy           : {accuracy:.4f}")
print(f"Precision          : {precision:.4f}")
print(f"Recall             : {recall:.4f}")
print(f"F1 Score           : {f1:.4f}")
print(f"Balanced Accuracy  : {balanced_acc:.4f}")
print(f"ROC-AUC            : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_landcover))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_landcover))

In [ ]:
# ============================================
# STEP 6: FEATURE IMPORTANCE
# ============================================

import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": encoded_feature_names,
    "Importance": rf_landcover.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print("============================================")
print("RANDOM FOREST FEATURE IMPORTANCE")
print("============================================")

print(feature_importance.to_string(index=False))

In [ ]:
# ============================================================
# 5-FOLD SPATIAL CROSS-VALIDATION
# ENVIRONMENT + LAND COVER
# ============================================================

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score
)

print("=" * 60)
print("5-FOLD SPATIAL CROSS-VALIDATION")
print("ENVIRONMENT + LAND COVER")
print("=" * 60)

# ------------------------------------------------------------
# IMPORTANT: Convert y to NumPy array
# This prevents pandas indexing errors with train_idx/test_idx
# ------------------------------------------------------------

X_cv = np.asarray(X)
y_cv = np.asarray(y)

print("X shape:", X_cv.shape)
print("y shape:", y_cv.shape)
print("Number of spatial blocks:", len(np.unique(spatial_blocks)))

# ------------------------------------------------------------
# Results storage
# ------------------------------------------------------------

spatial_cv_results = []

# ------------------------------------------------------------
# 5 spatial folds
# ------------------------------------------------------------

unique_blocks = np.unique(spatial_blocks)

# Reproducible shuffle of spatial blocks
rng = np.random.RandomState(42)
shuffled_blocks = rng.permutation(unique_blocks)

fold_blocks = np.array_split(shuffled_blocks, 5)

# ------------------------------------------------------------
# Run CV
# ------------------------------------------------------------

for fold in range(5):

    print()
    print("=" * 60)
    print(f"SPATIAL FOLD {fold + 1}")
    print("=" * 60)

    test_blocks = fold_blocks[fold]

    # All remaining blocks are training blocks
    train_blocks = np.concatenate(
        [fold_blocks[i] for i in range(5) if i != fold]
    )

    # Get row indices
    train_idx = np.where(np.isin(spatial_blocks, train_blocks))[0]
    test_idx = np.where(np.isin(spatial_blocks, test_blocks))[0]

    # --------------------------------------------------------
    # Split X
    # --------------------------------------------------------

    X_train_cv = X_cv[train_idx]
    X_test_cv = X_cv[test_idx]

    # --------------------------------------------------------
    # Split y
    # --------------------------------------------------------

    y_train_cv = y_cv[train_idx]
    y_test_cv = y_cv[test_idx]

    print("Training records:", len(train_idx))
    print("Testing records :", len(test_idx))
    print("Training blocks :", len(train_blocks))
    print("Testing blocks  :", len(test_blocks))

    # --------------------------------------------------------
    # Train Random Forest
    # --------------------------------------------------------

    rf_cv = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    rf_cv.fit(X_train_cv, y_train_cv)

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    y_pred_cv = rf_cv.predict(X_test_cv)
    y_prob_cv = rf_cv.predict_proba(X_test_cv)[:, 1]

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(y_test_cv, y_pred_cv)
    precision = precision_score(
        y_test_cv, y_pred_cv, zero_division=0
    )
    recall = recall_score(
        y_test_cv, y_pred_cv, zero_division=0
    )
    f1 = f1_score(
        y_test_cv, y_pred_cv, zero_division=0
    )
    balanced_acc = balanced_accuracy_score(
        y_test_cv, y_pred_cv
    )
    roc_auc = roc_auc_score(
        y_test_cv, y_prob_cv
    )

    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"ROC-AUC            : {roc_auc:.4f}")

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    spatial_cv_results.append({
        "Fold": fold + 1,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Balanced_Accuracy": balanced_acc,
        "ROC_AUC": roc_auc
    })


# ============================================================
# RESULTS SUMMARY
# ============================================================

spatial_cv_results_df = pd.DataFrame(spatial_cv_results)

print()
print("=" * 60)
print("SPATIAL CROSS-VALIDATION SUMMARY")
print("=" * 60)

display(spatial_cv_results_df)

# ------------------------------------------------------------
# Mean and standard deviation
# ------------------------------------------------------------

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Balanced_Accuracy",
    "ROC_AUC"
]

summary = pd.DataFrame({
    "Mean": spatial_cv_results_df[metric_columns].mean(),
    "Std": spatial_cv_results_df[metric_columns].std()
})

print()
print("Average performance across 5 spatial folds:")
display(summary)

5-FOLD SPATIAL CROSS-VALIDATION
ENVIRONMENT + LAND COVER


NameError: name 'X' is not defined

In [4]:
print("Variables currently available:")
print([v for v in globals().keys() if not v.startswith("_")])


Variables currently available:
['In', 'Out', 'get_ipython', 'exit', 'quit', 'pd', 'np', 'RandomForestClassifier', 'accuracy_score', 'precision_score', 'recall_score', 'f1_score', 'balanced_accuracy_score', 'roc_auc_score']


In [5]:
# ============================================================
# RELOAD FINAL MODELING DATASET
# ============================================================

import pandas as pd
import numpy as np

DATA_PATH = "./aviantrack_env/bahgoo_modeling_dataset.parquet"

df = pd.read_parquet(DATA_PATH)

print("=" * 60)
print("FINAL DATASET RELOADED")
print("=" * 60)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nClass distribution:")
print(df["PRESENCE"].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: './aviantrack_env/bahgoo_modeling_dataset.parquet'

In [6]:
import os

print("Current directory:", os.getcwd())

print("\nFiles/folders in /content:")
for item in os.listdir("/content"):
    print(item)

Current directory: /content

Files/folders in /content:
.config
drive
sample_data


In [7]:
import os

print("\nSearching for Bahgoo parquet files...\n")

matches = []

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".parquet") and "bahgoo" in file.lower():
            matches.append(os.path.join(root, file))

print("Found:", len(matches))

for path in matches:
    print(path)


Searching for Bahgoo parquet files...

Found: 0


In [8]:
import os

print("Searching Google Drive for project files...\n")

matches = []

for root, dirs, files in os.walk("/content/drive"):
    for file in files:
        if (
            "bahgoo" in file.lower()
            or "aviantrack" in file.lower()
        ):
            matches.append(os.path.join(root, file))

print("Files found:", len(matches))

for path in matches:
    print(path)

Searching Google Drive for project files...

Files found: 1
/content/drive/MyDrive/Colab Notebooks/AvianTrack.ipynb


In [9]:
import shutil
import os

source = "/content/drive/MyDrive/Colab Notebooks/AvianTrack.ipynb"
destination = "/content/AvianTrack.ipynb"

shutil.copy(source, destination)

print("Notebook restored successfully!")
print("Location:", destination)
print("File size:", os.path.getsize(destination) / (1024 * 1024), "MB")

Notebook restored successfully!
Location: /content/AvianTrack.ipynb
File size: 0.10111141204833984 MB


In [10]:
import json

with open("/content/AvianTrack.ipynb", "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("Number of cells:", len(notebook["cells"]))

for i, cell in enumerate(notebook["cells"]):
    cell_type = cell["cell_type"]
    source = "".join(cell.get("source", []))

    print(f"\n{'='*60}")
    print(f"CELL {i} | {cell_type}")
    print(f"{'='*60}")

    # Show first 500 characters of each cell
    print(source[:500])

Number of cells: 95

CELL 0 | code
!pip install rasterio geopandas pandas requests tqdm pyarrow


CELL 1 | code
import pandas as pd

presence = pd.read_csv("bahgoo_final_occurrences.csv")
background = pd.read_csv("bahgoo_background_points.csv")

print("PRESENCE:")
print(presence.shape)
print(presence.columns.tolist())

print("\nBACKGROUND:")
print(background.shape)
print(background.columns.tolist())

CELL 2 | code
!python 04b_extract_environmental_variables.py \
    --presence bahgoo_final_occurrences.csv \
    --background bahgoo_background_points.csv \
    --code bahgoo \
    --outdir ./aviantrack_env \
    --worldclim-res 2.5m

CELL 3 | code
import pandas as pd

# Convert presence data
presence = pd.read_csv("bahgoo_final_occurrences.csv")
presence.to_parquet("bahgoo_final_occurrences.parquet", index=False)

# Convert background data
background = pd.read_csv("bahgoo_background_points.csv")
background.to_parquet("bahgoo_background_points.parquet", index=False)

print("Conversion comp

In [11]:
import os
import shutil

print("Searching Google Drive for project files...")

matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if "bahgoo" in file.lower() or "aviantrack_env" in root.lower():
            matches.append(os.path.join(root, file))

print(f"Files found: {len(matches)}")

for path in matches:
    print(path)

Searching Google Drive for project files...
Files found: 0


In [12]:
import os

print("Current directory:", os.getcwd())

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith((".parquet", ".csv", ".geojson", ".zip")):
            print(os.path.join(root, file))

Current directory: /content
/content/drive/MyDrive/PROJECT DETAILS/fake_job_postings.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_test.csv


In [13]:
import os
import zipfile

print("Searching for project ZIP files...")

zip_files = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.lower().endswith(".zip"):
            zip_files.append(os.path.join(root, file))

print("ZIP files found:", len(zip_files))

for f in zip_files:
    print(f)

Searching for project ZIP files...
ZIP files found: 0


In [14]:
import json
import os

NOTEBOOK = "/content/AvianTrack.ipynb"

with open(NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("Number of cells:", len(nb["cells"]))

print("\nSearching notebook for dataset/file references...\n")

keywords = [
    "bahgoo",
    "modeling_dataset",
    "parquet",
    "csv",
    "files.zip",
    "aviantrack_env"
]

for i, cell in enumerate(nb["cells"]):
    text = "".join(cell.get("source", []))

    matches = [k for k in keywords if k.lower() in text.lower()]

    if matches:
        print("=" * 70)
        print("CELL:", i)
        print("MATCHES:", matches)
        print(text[:3000])

Number of cells: 95

Searching notebook for dataset/file references...

CELL: 1
MATCHES: ['bahgoo', 'csv']
import pandas as pd

presence = pd.read_csv("bahgoo_final_occurrences.csv")
background = pd.read_csv("bahgoo_background_points.csv")

print("PRESENCE:")
print(presence.shape)
print(presence.columns.tolist())

print("\nBACKGROUND:")
print(background.shape)
print(background.columns.tolist())
CELL: 2
MATCHES: ['bahgoo', 'csv', 'aviantrack_env']
!python 04b_extract_environmental_variables.py \
    --presence bahgoo_final_occurrences.csv \
    --background bahgoo_background_points.csv \
    --code bahgoo \
    --outdir ./aviantrack_env \
    --worldclim-res 2.5m
CELL: 3
MATCHES: ['bahgoo', 'parquet', 'csv']
import pandas as pd

# Convert presence data
presence = pd.read_csv("bahgoo_final_occurrences.csv")
presence.to_parquet("bahgoo_final_occurrences.parquet", index=False)

# Convert background data
background = pd.read_csv("bahgoo_background_points.csv")
background.to_parquet("bahgoo_

In [15]:
!du -h --max-depth=2 /content/aviantrack_env 2>/dev/null | sort -h | tail -30

In [16]:
!du -h --max-depth=2 /content 2>/dev/null | sort -h | tail -30

4.0K	/content/drive/.shortcut-targets-by-id
8.0K	/content/.config/configurations
12K	/content/drive/.Trash-0
28K	/content/drive/.Encrypted
92K	/content/.config/logs
148K	/content/.config
55M	/content/sample_data
182M	/content/drive
182M	/content/drive/MyDrive
236M	/content


In [17]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G  108G     0 100% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.3G  696M  65% /usr/sbin/docker-init
/dev/sda1       114G  109G  5.4G  96% /kaggle/input
tmpfs           6.4G  556K  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
drive            15G   15G     0 100% /content/drive


In [18]:
!du -h --max-depth=1 / 2>/dev/null | sort -h | tail -20

4.0K	/lib.usr-is-merged
4.0K	/media
4.0K	/mnt
4.0K	/sbin.usr-is-merged
4.0K	/srv
8.0K	/kaggle
20K	/home
100K	/run
112K	/tmp
3.8M	/etc
20M	/python-apt
51M	/datalab
88M	/opt
116M	/var
236M	/content
1.1G	/tools
3.3G	/root
16G	/usr
88G	/bin.usr-is-merged
108G	/


In [19]:
!du -h --max-depth=2 /bin.usr-is-merged 2>/dev/null | sort -h | tail -30

88G	/bin.usr-is-merged
88G	/bin.usr-is-merged/dev


In [20]:
!ls -lah /bin.usr-is-merged | head -30

total 12K
drwxr-xr-x 1 root root 4.0K Sep 13 14:48 .
drwxr-xr-x 1 root root 4.0K Sep 13 15:02 ..
drwxr-xr-x 2 root root 4.0K Sep 13 14:43 dev


In [21]:
!lsof +L1 2>/dev/null | head -50


In [22]:
!df -h /bin.usr-is-merged/dev

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G  108G     0 100% /


In [23]:
!mount | grep -E '/bin|/dev'

tmpfs on /dev type tmpfs (rw,nosuid,size=65536k,mode=755)
devpts on /dev/pts type devpts (rw,nosuid,noexec,relatime,gid=5,mode=620,ptmxmode=666)
mqueue on /dev/mqueue type mqueue (rw,nosuid,nodev,noexec,relatime)
shm on /dev/shm type tmpfs (rw,nosuid,nodev,noexec,relatime,size=5939200k)
/dev/dm-0 on /usr/sbin/docker-init type ext2 (ro,relatime)
/dev/sda1 on /kaggle/input type ext4 (ro,nosuid,nodev,noexec,relatime,commit=30)
/dev/sda1 on /etc/resolv.conf type ext4 (rw,nosuid,nodev,relatime,commit=30)
/dev/sda1 on /etc/hostname type ext4 (rw,nosuid,nodev,relatime,commit=30)
/dev/sda1 on /etc/hosts type ext4 (rw,nosuid,nodev,relatime,commit=30)


In [24]:
!stat /bin.usr-is-merged/dev

  File: /bin.usr-is-merged/dev
  Size: 4096      	Blocks: 8          IO Block: 4096   directory
Device: 0,52	Inode: 2889194     Links: 2
Access: (0755/drwxr-xr-x)  Uid: (    0/    root)   Gid: (    0/    root)
Access: 2026-09-13 15:07:03.118296792 +0000
Modify: 2026-09-13 14:43:51.399932143 +0000
Change: 2026-09-13 15:02:02.322252436 +0000
 Birth: 2026-09-13 14:48:14.406032680 +0000


In [25]:
!du -xhd1 /bin.usr-is-merged/dev 2>/dev/null

88G	/bin.usr-is-merged/dev


In [26]:
!find /content /tmp /root -type f \( -name "*.tif" -o -name "*.zip" -o -name "*.parquet" \) -printf '%s %p\n' 2>/dev/null | sort -nr | head -30

520 /root/.julia/packages/TiffImages/VlJh4/src/precomp-tifs/images/Float64_RGB_(2, 2, 2).tif
424 /root/.julia/packages/TiffImages/VlJh4/src/precomp-tifs/images/Float32_RGB_(2, 2, 2).tif
376 /root/.julia/packages/TiffImages/VlJh4/src/precomp-tifs/images/N0f16_RGB_(2, 2, 2).tif
368 /root/.julia/packages/TiffImages/VlJh4/src/precomp-tifs/images/Float64_Gray_(2, 2, 2).tif
352 /root/.julia/packages/TiffImages/VlJh4/src/precomp-tifs/images/N0f8_RGB_(2, 2, 2).tif
336 /root/.julia/packages/TiffImages/VlJh4/src/precomp-tifs/images/Float32_Gray_(2, 2, 2).tif
320 /root/.julia/packages/TiffImages/VlJh4/src/precomp-tifs/images/N0f16_Gray_(2, 2, 2).tif
312 /root/.julia/packages/TiffImages/VlJh4/src/precomp-tifs/images/N0f8_Gray_(2, 2, 2).tif
264 /root/.julia/packages/TiffImages/VlJh4/src/precomp-tifs/images/Float64_RGB_(2, 2).tif
216 /root/.julia/packages/TiffImages/VlJh4/src/precomp-tifs/images/Float32_RGB_(2, 2).tif
192 /root/.julia/packages/TiffImages/VlJh4/src/precomp-tifs/images/N0f16_RGB_(2, 2

In [27]:
!ls -ld /bin /bin.usr-is-merged /bin.usr-is-merged/dev

lrwxrwxrwx 1 root root    7 Apr 22  2024 /bin -> usr/bin
drwxr-xr-x 1 root root 4096 Sep 13 14:48 /bin.usr-is-merged
drwxr-xr-x 2 root root 4096 Sep 13 14:43 /bin.usr-is-merged/dev


In [28]:
!readlink -f /bin.usr-is-merged/dev

/bin.usr-is-merged/dev


In [29]:
!findmnt -T /bin.usr-is-merged/dev

TARGET SOURCE FSTYPE OPTIONS
/      overlay
              overla rw,relatime,lowerdir=/var/lib/docker/overlay2/l/QYPHGUVW3OS


In [30]:
!find /bin.usr-is-merged/dev -maxdepth 2 -type f -printf '%s %p\n' 2>/dev/null | head -30

0 /bin.usr-is-merged/dev/null
0 /bin.usr-is-merged/dev/random
0 /bin.usr-is-merged/dev/full
0 /bin.usr-is-merged/dev/fuse
93821927424 /bin.usr-is-merged/dev/urandom
0 /bin.usr-is-merged/dev/zero


In [31]:
!df -h /

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G  108G     0 100% /


In [32]:
!find /content -maxdepth 3 -type f | head -50

/content/.config/.last_update_check.json
/content/.config/config_sentinel
/content/.config/.last_survey_prompt.yaml
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/active_config
/content/.config/default_configs.db
/content/.config/.last_opt_in_prompt.yaml
/content/.config/gce
/content/.config/configurations/config_default
/content/bahgoo_final_occurrences.csv
/content/AvianTrack.ipynb
/content/04b_extract_environmental_variables.py
/content/bahgoo_background_points.csv
/content/04c_add_landcover.py
/content/bahgoo_accessible_area.geojson
/content/drive/MyDrive/in.gov.abc-ABCID-364673441752.pdf
/content/drive/MyDrive/IMG-20230823-WA0001 (1).jpg
/content/drive/MyDrive/IMG-20230823-WA0001.jpg
/content/drive/MyDrive/Document from Manish
/content/drive/MyDrive/FormSeven.pdf
/content/drive/MyDrive/Introduction to Cloud Computing.pdf
/content/drive/MyDrive/IMG-20250727-WA0007~2 (2).jpg
/content/drive/MyDrive/IMG-20250727-WA0007~2 (1).jpg
/conte

In [34]:
!find / -type f \( -name "bahgoo_modeling_dataset.parquet" -o -name "*modeling*dataset*" \) 2>/dev/null

In [35]:
!du -h --max-depth=2 /content/drive/MyDrive 2>/dev/null | sort -h | tail -30

4.5K	/content/drive/MyDrive/Google Earth
666K	/content/drive/MyDrive/Colab Notebooks
46M	/content/drive/MyDrive/PROJECT VIDEO
49M	/content/drive/MyDrive/PROJECT DETAILS
182M	/content/drive/MyDrive


In [36]:
!df -h /

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G  108G     0 100% /


In [37]:
!find /content -maxdepth 3 -type f | head -50

/content/.config/.last_update_check.json
/content/.config/config_sentinel
/content/.config/.last_survey_prompt.yaml
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/active_config
/content/.config/default_configs.db
/content/.config/.last_opt_in_prompt.yaml
/content/.config/gce
/content/.config/configurations/config_default
/content/bahgoo_final_occurrences.csv
/content/AvianTrack.ipynb
/content/04b_extract_environmental_variables.py
/content/bahgoo_background_points.csv
/content/04c_add_landcover.py
/content/bahgoo_accessible_area.geojson
/content/drive/MyDrive/in.gov.abc-ABCID-364673441752.pdf
/content/drive/MyDrive/IMG-20230823-WA0001 (1).jpg
/content/drive/MyDrive/IMG-20230823-WA0001.jpg
/content/drive/MyDrive/Document from Manish
/content/drive/MyDrive/FormSeven.pdf
/content/drive/MyDrive/Introduction to Cloud Computing.pdf
/content/drive/MyDrive/IMG-20250727-WA0007~2 (2).jpg
/content/drive/MyDrive/IMG-20250727-WA0007~2 (1).jpg
/conte

In [1]:
import os

print("=== PARQUET FILES ===")
for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.endswith(".parquet"):
            print(os.path.join(root, f))

print("\n=== TIFF FILES ===")
count = 0
for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.endswith((".tif", ".tiff")):
            print(os.path.join(root, f))
            count += 1

print("\nTotal TIFF files:", count)

=== PARQUET FILES ===

=== TIFF FILES ===

Total TIFF files: 0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

PROJECT_DIR = "/content/drive/MyDrive/AvianTrack"

os.makedirs(PROJECT_DIR, exist_ok=True)

print("Project folder:")
print(PROJECT_DIR)

Project folder:
/content/drive/MyDrive/AvianTrack


In [4]:
import shutil
import os

files_to_save = [
    "/content/AvianTrack.ipynb",
    "/content/bahgoo_final_occurrences.csv",
    "/content/bahgoo_background_points.csv",
    "/content/bahgoo_accessible_area.geojson",
    "/content/04b_extract_environmental_variables.py",
    "/content/04c_add_landcover.py"
]

for src in files_to_save:
    if os.path.exists(src):
        shutil.copy2(src, PROJECT_DIR)
        print("✅ Saved:", os.path.basename(src))
    else:
        print("❌ Missing:", src)

❌ Missing: /content/AvianTrack.ipynb
✅ Saved: bahgoo_final_occurrences.csv
✅ Saved: bahgoo_background_points.csv
✅ Saved: bahgoo_accessible_area.geojson
✅ Saved: 04b_extract_environmental_variables.py
✅ Saved: 04c_add_landcover.py
